In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "AntonV/mamba2-130m-hf"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16)
model.eval()

/vol/fob-vol3/mi20/taquahie/Study_Project/chat_state_management/.venv/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] The fast path is not available because one of `(selective_state_update, causal_conv1d_fn, causal_conv1d_update)` is None. Falling back to the naive implementation. To install follow https://github.com/state-spaces/mamba/#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 218/218 [00:00<00:00, 2486.00it/s]


Mamba2ForCausalLM(
  (backbone): Mamba2Model(
    (embeddings): Embedding(50288, 768)
    (layers): ModuleList(
      (0-23): 24 x Mamba2Block(
        (norm): Mamba2RMSNorm()
        (mixer): Mamba2Mixer(
          (act): SiLUActivation()
          (conv1d): Conv1d(1792, 1792, kernel_size=(4,), stride=(1,), padding=(3,), groups=1792)
          (in_proj): Linear(in_features=768, out_features=3352, bias=False)
          (norm): MambaRMSNormGated()
          (out_proj): Linear(in_features=1536, out_features=768, bias=False)
        )
      )
    )
    (norm_f): Mamba2RMSNorm()
  )
  (lm_head): Linear(in_features=768, out_features=50288, bias=False)
)

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(torch.__version__)
print(torch.version.cuda)

Using device: cuda
2.12.0+cu130
13.0


In [3]:
from datasets import load_dataset
dataset_name = "nvidia/Nemotron-RL-Instruction-Following-MultiTurnChat-v1"
split = "train"
dataset = load_dataset(dataset_name, split=split)

In [4]:
from src import model_loader, state_utils, evaluate as evaluate_module, data, autoencoder, plot, utils
model.to(device)

Mamba2ForCausalLM(
  (backbone): Mamba2Model(
    (embeddings): Embedding(50288, 768)
    (layers): ModuleList(
      (0-23): 24 x Mamba2Block(
        (norm): Mamba2RMSNorm()
        (mixer): Mamba2Mixer(
          (act): SiLUActivation()
          (conv1d): Conv1d(1792, 1792, kernel_size=(4,), stride=(1,), padding=(3,), groups=1792)
          (in_proj): Linear(in_features=768, out_features=3352, bias=False)
          (norm): MambaRMSNormGated()
          (out_proj): Linear(in_features=1536, out_features=768, bias=False)
        )
      )
    )
    (norm_f): Mamba2RMSNorm()
  )
  (lm_head): Linear(in_features=768, out_features=50288, bias=False)
)

In [5]:
sessions = data.extract_sessions(dataset)
states = []
list_of_snapshots = []
num_turn = 0
for session in sessions:
    snapshots = data.build_turn_snapshots(session)
    list_of_snapshots.append(snapshots)
    num_turn += len(snapshots)

print(len(list_of_snapshots), num_turn)

2011 30631


In [6]:
config = utils.read_config("configs/config1.yaml")
config["data"]["max_length"] = 1024

paths = config["paths"]
output_dir = paths["output_dir"]
text_history_dir = paths["text_history_dir"]+"/history.txt"
state_dir = paths["state_dir"]+"/state.pt"
plot_dir = paths["plot_dir"]
experiment2_path = output_dir + "/experiment2/experiment2.csv"
max_seq_length = config["data"]["max_length"]
experiment1_path = output_dir + "/experiment1/experiment1.csv"

In [7]:
import copy
import yaml
import torch
import pandas as pd
import csv
import time
import numpy as np
NUM_RUNS = 5

In [8]:
def run_baseline(model, tokenizer, snapshots, device, text_history_dir):
    output_data = {}
    for snap in snapshots:
        turn_id = snap["turn_id"]
        history_text = ""
        input_text = snap["new_text"]

        if turn_id > 0:
            history_text = utils.load_text(text_history_dir)

        combined_text = utils.concatenate_texts([history_text, snap["new_text"]])

        encoded_input = tokenizer(combined_text, max_length=max_seq_length, truncation=True, return_tensors="pt")
        truncated_input_ids = encoded_input["input_ids"]
        truncated_combined_text = tokenizer.decode(truncated_input_ids[0], skip_special_tokens=True)

        _, baseline_latency, baseline_ppl = evaluate_module.evaluate_baseline(
            model,
            tokenizer,
            truncated_combined_text,
            input_text,
            device=device
        )
        if snap["role"] == "assistant":
            baseline_size_kb = utils.get_memory_size_kb(text_history_dir)
            output_data[turn_id] = {
                "baseline_latency": baseline_latency,
                "baseline_size_kb": baseline_size_kb,
                "baseline_ppl": baseline_ppl
            }
        utils.save_text(truncated_combined_text, text_history_dir)
    return output_data

def run_state_management(model, tokenizer, snapshots, device, state_dir):
    output_data = {}
    for snap in snapshots:
        turn_id = snap["turn_id"]

        if turn_id %2!= 0:
            history_text = snap["history_text"]
            history_ids = tokenizer(history_text, return_tensors="pt").to(device)
            state_output = model(history_ids["input_ids"], use_cache=True)
        else:
            previous_state = state_utils.load_state(state_dir)
            state = {}
            if type(previous_state) == list:
                for layer_idx, states in enumerate(previous_state):
                    state[layer_idx] = states
            else:
                state = previous_state
            state_output, state_latency, state_ppl = evaluate_module.evaluate_injeted_mode(
                model,
                tokenizer,
                snap["new_text"],
                state,
                device=device
            )

        new_state = state_utils.save_recurrent_states(state_output.cache_params)
        state_utils.save_state(new_state, state_dir)
        if snap["role"] == "assistant":
            state_size_kb = utils.get_memory_size_kb(state_dir)
            output_data[turn_id] = {
                "state_latency": state_latency,
                "state_size_kb": state_size_kb,
                "state_ppl": state_ppl
            }
    return output_data

In [20]:
def run_experiment_1():
    experiment1_path = output_dir + "/experiment1/experiment1.csv"
    baseline_output = {}
    state_output = {}
    for _ in range(NUM_RUNS):
        torch.cuda.empty_cache()
        print(f"Run {_+1}/{NUM_RUNS}...")
        print("starting baseline run...")
        baseline_data = run_baseline(model, tokenizer, snapshots, device, text_history_dir)
        print("starting state management run...")
        state_data = run_state_management(model, tokenizer, snapshots, device, state_dir)
        print("aggregating results...")
        for turn_id in baseline_data:
            if turn_id not in baseline_output:
                baseline_output[turn_id] = {}
                baseline_output[turn_id]["baseline_latency"] = 0
                baseline_output[turn_id]["baseline_size_kb"] = 0
                baseline_output[turn_id]["baseline_ppl"] = 0
            baseline_output[turn_id]["baseline_latency"] += baseline_data[turn_id]["baseline_latency"]
            baseline_output[turn_id]["baseline_size_kb"] += baseline_data[turn_id]["baseline_size_kb"]
            baseline_output[turn_id]["baseline_ppl"] += baseline_data[turn_id]["baseline_ppl"]
            
            if turn_id not in state_output:
                state_output[turn_id] = {}
                state_output[turn_id]["state_latency"] = 0
                state_output[turn_id]["state_size_kb"] = 0
                state_output[turn_id]["state_ppl"] = 0
            state_output[turn_id]["state_latency"] += state_data[turn_id]["state_latency"]
            state_output[turn_id]["state_size_kb"] += state_data[turn_id]["state_size_kb"]
            state_output[turn_id]["state_ppl"] += state_data[turn_id]["state_ppl"]

        print(f"Completed run {_+1}/{NUM_RUNS}")

    for turn_id in baseline_output:
        baseline_output[turn_id]["baseline_latency"] /= NUM_RUNS
        baseline_output[turn_id]["baseline_size_kb"] /= NUM_RUNS
        baseline_output[turn_id]["baseline_ppl"] /= NUM_RUNS

        state_output[turn_id]["state_latency"] /= NUM_RUNS
        state_output[turn_id]["state_size_kb"] /= NUM_RUNS
        state_output[turn_id]["state_ppl"] /= NUM_RUNS

    with open(experiment1_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["turn", "baseline_latency", "state_latency", "txt_size_kb", "pt_size_kb", "baseline_ppl", "state_ppl"])
        for turn_id in baseline_output:
            writer.writerow([
                turn_id,
                baseline_output[turn_id]["baseline_latency"],
                state_output[turn_id]["state_latency"],
                baseline_output[turn_id]["baseline_size_kb"],
                state_output[turn_id]["state_size_kb"],
                baseline_output[turn_id]["baseline_ppl"],
                state_output[turn_id]["state_ppl"]
            ])

    df = pd.read_csv(experiment1_path)
    latency_df = df[["turn", "baseline_latency", "state_latency"]]
    size_df = df[["turn", "txt_size_kb", "pt_size_kb"]]
    plot.plot_memory_growth(size_df, plot_dir + "/experiment1/memory_growth.png")
    plot.plot_latency_comparison(latency_df, plot_dir + "/experiment1/latency_comparison.png")
    plot.plot_ppl_comparison(df[["turn", "baseline_ppl", "state_ppl"]], plot_dir + "/experiment1/perplexity_comparison.png")

In [21]:
run_experiment_1()

Run 1/5...
starting baseline run...
starting state management run...
aggregating results...
Completed run 1/5
Run 2/5...
starting baseline run...
starting state management run...
aggregating results...
Completed run 2/5
Run 3/5...
starting baseline run...
starting state management run...
aggregating results...
Completed run 3/5
Run 4/5...
starting baseline run...
starting state management run...
aggregating results...
Completed run 4/5
Run 5/5...
starting baseline run...
starting state management run...
aggregating results...
Completed run 5/5


In [ ]:
def run_autoencoder(model, tokenizer, snapshots, device, state_dir, ae_list):
    output_data = {}
    for ae in ae_list:
        ae.eval()
        ae.to(device)
    for snap in snapshots:
        turn_id = snap["turn_id"]

        if turn_id == 0:
            state_output, state_latency, state_ppl = evaluate_module.evaluate_baseline(
                model,
                tokenizer,
                snap["history_text"],
                snap["new_text"],
                device=device
            )
        else:
            compressed_state = state_utils.load_state(state_dir)

            decompressed_state = {}
            for layer_idx, latent in compressed_state.items():
                latent = latent.to(device)                          # [heads, latent_dim]
                latent = latent.unsqueeze(0)                        # [1, heads, latent_dim]
                reconstructed = ae_list[layer_idx].decoder(latent)  # [1, heads, head_dim, d_state]
                # print(f"layer: {layer_idx}, reconstructed shape: {reconstructed.shape}")
                reconstructed = reconstructed.view(1, 24, 64, 128)
                decompressed_state[layer_idx] = reconstructed
                
            state_output, state_latency, state_ppl = evaluate_module.evaluate_injeted_mode(
                model,
                tokenizer,
                snap["new_text"],
                decompressed_state,
                device=device
            )

        raw_states = state_utils.save_recurrent_states(state_output.cache_params)
        compressed = {}
        for layer_idx, state in raw_states.items():
            state = state.to(device)                                # [1, heads, head_dim, d_state]
            latent = ae_list[layer_idx].encoder(
                state.view(1, 24, -1)                               # [1, heads, head_dim*d_state]
            )                                                       # [1, heads, latent_dim]
            compressed[layer_idx]=(latent.squeeze(0).cpu())              # [heads, latent_dim]

        state_utils.save_state(compressed, state_dir)

        if snap["role"] == "assistant":
            state_size_kb = utils.get_memory_size_kb(state_dir)
            output_data[turn_id] = {
                "state_latency": state_latency,
                "state_size_kb": state_size_kb,
                "state_ppl": state_ppl
            }

    return output_data

In [ ]:
def run_experiment_2(
    model, tokenizer, snapshots, ae_experiments,  # ae_experiments: {latent_dim: ae_list}
    output_dir, experiment_2_benchmark_path, plot_dir,
    device
):
    # Write header once
    with open(experiment_2_benchmark_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "turn", "state_latency", "compressed_latency",
            "state_ppl", "compressed_ppl",
            "original_size_kb", "compressed_size_kb",
            "autoencoder_latent_dim"
        ])

    print("Running original state management...")
    original_dir     = f"{output_dir}/state_original.pt"
    
    original_results = run_state_management(
        model, tokenizer, snapshots, device, original_dir
    )

    for latent_dim, ae_list in ae_experiments.items():
        print(f"\n{'='*50}")
        print(f"Running AE latent_dim={latent_dim}")
        print(f"{'='*50}")

        compress_dir = f"{output_dir}/state_compressed_{latent_dim}.pt"

        ae_results = run_autoencoder(
            model, tokenizer, snapshots, device, compress_dir, ae_list
        )

        with open(experiment_2_benchmark_path, "a", newline="") as f:
            writer = csv.writer(f)
            for turn_id in original_results:
                if turn_id not in ae_results:
                    continue
                orig = original_results[turn_id]
                comp = ae_results[turn_id]
                writer.writerow([
                    turn_id,
                    orig["state_latency"], comp["state_latency"],
                    orig["state_ppl"],     comp["state_ppl"],
                    orig["state_size_kb"], comp["state_size_kb"],
                    latent_dim
                ])
        print(f"Finish running AE latent_dim={latent_dim}")
        print(f"{'='*50}")
        torch.cuda.empty_cache()

    df = pd.read_csv(experiment_2_benchmark_path)
    plot.plot_perplexity_comparison(df, plot_dir)
    plot.plot_latency_comparison_exp2(df, plot_dir)
    plot.plot_memory_growth_exp2(df, plot_dir)
    return df

In [ ]:
import torch.nn as nn
latent_dims = [32, 64, 128]
num_layers = 24

ae_experiments = {
    ld: nn.ModuleList([
        autoencoder.Autoencoder(head_dim=64, d_state=128, hidden_dim=ld)
        for _ in range(num_layers)
    ])
    for ld in latent_dims
}

In [ ]:
import os
save_dir = "autoencoders"

In [ ]:
for latent_dim, ae_list in ae_experiments.items():
    for layer_idx in range(num_layers):
        path =  os.path.join(save_dir, f"latent_dim_{latent_dim}/layer_{layer_idx}/autoencoder.pt")
        ae_list[layer_idx].load_state_dict(
            torch.load(
                path,
                map_location="cpu")
        )
        ae_list[layer_idx].eval()

In [ ]:
# Convert all autoencoders to float16 to match model dtype
for latent_dim, ae_list in ae_experiments.items():
    ae_list.to(torch.float16)
    ae_list.to(device)

In [ ]:
df = run_experiment_2(
    model, tokenizer, snapshots, ae_experiments,
    output_dir + "/experiment2",
    experiment2_path,
    plot_dir + "/experiment2",
    device
)

Running original state management...

Running AE latent_dim=32
Finish running AE latent_dim=32

Running AE latent_dim=64
Finish running AE latent_dim=64

Running AE latent_dim=128
Finish running AE latent_dim=128
